In [ ]:

!pip install groq together \
            langchain langchain-community langchain-groq \
            langchain-together langchain-text-splitters \
            faiss-cpu sentence-transformers \
            pypdf pandas tiktoken langchain-huggingface --quiet


GROQ_API_KEY     = ""
TOGETHER_API_KEY = ""
MODEL_A = {"name": "Llama-3",  "provider": "groq",     "model": "llama-3.1-8b-instant"}
MODEL_B = {"name": "Mixtral",  "provider": "groq",     "model": "mixtral-8x7b-32768"}
MODEL_C = {"name": "Gemma-2",  "provider": "together", "model": "google/gemma-2-9b-it"}

YOUR_QUESTION = "What are the key findings or main points in this document?"

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K_CHUNKS  = 4
DEBATE_ROUNDS = 2

import os, re, time, textwrap
os.environ["GROQ_API_KEY"]     = GROQ_API_KEY
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY

# LangChain — text splitting (new correct package)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain — document loaders
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader

# LangChain — embeddings + vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# LangChain — LLM wrappers
from langchain_groq import ChatGroq
from langchain_together import ChatTogether

# LangChain — prompts + output parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("✓ All imports successful.")


from google.colab import files as colab_files

print("\n📂 Upload your file (PDF, TXT, or CSV)...")
uploaded = colab_files.upload()
FILEPATH = list(uploaded.keys())[0]
print(f"✓ Uploaded: {FILEPATH}")



class RAGPipeline:

    def __init__(self, filepath: str):
        self.filepath = filepath
        self._build()

    def _load(self):
        ext = self.filepath.rsplit(".", 1)[-1].lower()
        if ext == "pdf":
            return PyPDFLoader(self.filepath).load()
        elif ext == "csv":
            return CSVLoader(self.filepath).load()
        else:
            return TextLoader(self.filepath, encoding="utf-8").load()

    def _build(self):
        print("\n🔧 Building RAG pipeline...")

        print("   [1/4] Loading document...")
        docs = self._load()
        print(f"         {len(docs)} page(s) loaded.")

        print(f"   [2/4] Splitting into chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})...")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""],
        )
        self.chunks = splitter.split_documents(docs)
        print(f"         {len(self.chunks)} chunks created.")

        print("   [3/4] Embedding with all-MiniLM-L6-v2 (runs locally)...")
        embeddings = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={"device": "cpu"},
        )

        print("   [4/4] Building FAISS vector index...")
        self.vectordb = FAISS.from_documents(self.chunks, embeddings)
        self.retriever = self.vectordb.as_retriever(
            search_type="similarity",
            search_kwargs={"k": TOP_K_CHUNKS},
        )
        print("✓ RAG pipeline ready.\n")

    def retrieve(self, query: str):
        docs = self.retriever.invoke(query)
        context = "\n\n---\n\n".join(d.page_content for d in docs)
        sources  = [
            f"page {d.metadata.get('page', '?')}" for d in docs
        ]
        return context, sources




SYSTEM_MSG = (
    "You are a rigorous analytical agent. "
    "Reason step-by-step, ground every claim in the provided context, "
    "and flag anything you are uncertain about."
)

INITIAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_MSG),
    ("human",
     "RETRIEVED CONTEXT:\n{context}\n\n"
     "QUESTION: {question}\n\n"
     "Give a thorough evidence-based answer using only the context above.\n"
     "End with:\nCONFIDENCE: <0-100>\nSUMMARY: <one sentence>"),
])

DEBATE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_MSG),
    ("human",
     "RETRIEVED CONTEXT:\n{context}\n\n"
     "QUESTION: {question}\n\n"
     "OTHER AGENTS' RESPONSES:\n{others}\n\n"
     "1. Point out errors or unsupported claims in the other responses.\n"
     "2. State what you agree with.\n"
     "3. Write your REVISED final answer grounded in the context.\n"
     "End with:\nCONFIDENCE: <0-100>\nSUMMARY: <one sentence>"),
])


def make_llm(cfg: dict):
    if cfg["provider"] == "groq":
        return ChatGroq(
            model=cfg["model"],
            temperature=0.7,
            max_tokens=900,
            api_key=GROQ_API_KEY,
        )
    return ChatTogether(
        model=cfg["model"],
        temperature=0.7,
        max_tokens=900,
        together_api_key=TOGETHER_API_KEY,
    )


class Agent:
    def __init__(self, cfg: dict):
        self.name   = cfg["name"]
        self.llm    = make_llm(cfg)
        self.parser = StrOutputParser()

    def _call(self, prompt, inputs: dict) -> str:
        chain = prompt | self.llm | self.parser
        for attempt in range(3):
            try:
                return chain.invoke(inputs)
            except Exception as e:
                if attempt == 2:
                    return f"[ERROR {self.name}: {e}]"
                time.sleep(2 ** attempt)

    def initial(self, context: str, question: str) -> str:
        return self._call(INITIAL_PROMPT, {"context": context, "question": question})

    def debate(self, context: str, question: str, others: list) -> str:
        others_block = "\n\n".join(f"=== {n} ===\n{r}" for n, r in others)
        return self._call(DEBATE_PROMPT, {
            "context": context,
            "question": question,
            "others": others_block,
        })


def get_confidence(text: str) -> float:
    m = re.search(r"CONFIDENCE[:\s]+(\d{1,3})", text, re.IGNORECASE)
    return float(m.group(1)) if m else 60.0

def get_summary(text: str) -> str:
    m = re.search(r"SUMMARY[:\s]+(.+?)(\n|$)", text, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return parts[-1] if parts else text[:200]

def clean(text: str) -> str:
    return re.sub(r"(CONFIDENCE|SUMMARY)[:\s]+.+?(\n|$)", "",
                  text, flags=re.IGNORECASE).strip()

def jaccard(a: str, b: str) -> float:
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

def score_all(responses: dict) -> dict:
    names = list(responses.keys())
    self_conf = {n: get_confidence(t) for n, t in responses.items()}

    pairs = [
        (names[i], names[j], jaccard(responses[names[i]], responses[names[j]]))
        for i in range(len(names)) for j in range(i+1, len(names))
    ]
    avg_agree = sum(s for _, _, s in pairs) / len(pairs) if pairs else 0.5
    low_agree = [(a, b) for a, b, s in pairs if s < 0.2]

    per_agent = {
        n: round(0.6 * self_conf[n] + 0.4 * avg_agree * 100)
        for n in names
    }
    return {
        "overall":      round(sum(per_agent.values()) / len(per_agent)),
        "per_agent":    per_agent,
        "agreement":    round(avg_agree * 100),
        "disagreements": low_agree,
    }




def run_debate(question: str, filepath: str):

    # RAG
    rag = RAGPipeline(filepath)
    print(f"🔍 Retrieving top-{TOP_K_CHUNKS} chunks for the question...")
    context, sources = rag.retrieve(question)
    print(f"   Sources: {', '.join(sources)}\n")

    # Build agents
    agents = [Agent(MODEL_A), Agent(MODEL_B), Agent(MODEL_C)]

    # Phase 1 — initial responses
    print("🤖  Phase 1 — Initial responses")
    print("─" * 60)
    current = {}
    for ag in agents:
        print(f"  ⏳ {ag.name} thinking…", end=" ", flush=True)
        resp = ag.initial(context, question)
        current[ag.name] = resp
        print(f"done  [confidence: {get_confidence(resp):.0f}%]")

    # Phase 2 — debate rounds
    for rnd in range(1, DEBATE_ROUNDS + 1):
        print(f"\n⚔️   Debate round {rnd}/{DEBATE_ROUNDS}")
        print("─" * 60)
        revised = {}
        for ag in agents:
            others = [(n, r) for n, r in current.items() if n != ag.name]
            print(f"  ⏳ {ag.name} critiquing peers…", end=" ", flush=True)
            resp = ag.debate(context, question, others)
            revised[ag.name] = resp
            print(f"done  [confidence: {get_confidence(resp):.0f}%]")
        current = revised

    # Score
    scores    = score_all(current)
    best_name = max(scores["per_agent"], key=scores["per_agent"].get)
    best_text = clean(current[best_name])

    # Print output
    W = 60
    print("\n" + "═"*W)
    print(f"✅  BEST ANSWER  (from {best_name})")
    print("═"*W)
    print(textwrap.fill(best_text, width=W))

    print("\n" + "─"*W)
    flag = "  ⚠️ LOW — verify manually" if scores["overall"] < 70 else "  ✓"
    print(f"📊  OVERALL CONFIDENCE   : {scores['overall']}%{flag}")
    print(f"🤝  INTER-AGENT AGREEMENT: {scores['agreement']}%\n")

    print("👥  PER-AGENT CONFIDENCE:")
    for name, sc in scores["per_agent"].items():
        bar = "█"*(sc//10) + "░"*(10 - sc//10)
        sel = "  ← selected" if name == best_name else ""
        print(f"    {name:<18} {bar}  {sc}%{sel}")

    if scores["disagreements"]:
        print("\n❌  SIGNIFICANT DISAGREEMENTS:")
        for a, b in scores["disagreements"]:
            print(f"    {a}  ↔  {b}  (very low overlap)")

    print("\n📝  ONE-LINE SUMMARIES:")
    for ag in agents:
        print(f"    {ag.name}: {get_summary(current[ag.name])}")

    print("\n🗂️   RAG CHUNKS USED:")
    for s in sources:
        print(f"    • {s}")

    print("\n🧠  WHY THIS ANSWER:")
    reason = (
        f"{best_name} scored highest ({scores['per_agent'][best_name]}%) "
        f"combining self-reported confidence and inter-agent agreement "
        f"({scores['agreement']}% token overlap). "
        f"All agents answered from the same {TOP_K_CHUNKS} retrieved chunks."
    )
    print(textwrap.fill("    " + reason, width=W))
    print("═"*W + "\n")


run_debate(question=YOUR_QUESTION, filepath=FILEPATH)

✓ All imports successful.

📂 Upload your file (PDF, TXT, or CSV)...


Saving AII(1).pdf to AII(1) (2).pdf
✓ Uploaded: AII(1) (2).pdf

🔧 Building RAG pipeline...
   [1/4] Loading document...
         2 page(s) loaded.
   [2/4] Splitting into chunks (size=500, overlap=50)...
         13 chunks created.
   [3/4] Embedding with all-MiniLM-L6-v2 (runs locally)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   [4/4] Building FAISS vector index...
✓ RAG pipeline ready.

🔍 Retrieving top-4 chunks for the question...
   Sources: page 1, page 1, page 1, page 1

🤖  Phase 1 — Initial responses
────────────────────────────────────────────────────────────
  ⏳ Llama-3 thinking… done  [confidence: 100%]
  ⏳ Mixtral thinking… done  [confidence: 60%]
  ⏳ Gemma-2 thinking… done  [confidence: 60%]

⚔️   Debate round 1/2
────────────────────────────────────────────────────────────
  ⏳ Llama-3 critiquing peers… done  [confidence: 90%]
  ⏳ Mixtral critiquing peers… done  [confidence: 60%]
  ⏳ Gemma-2 critiquing peers… done  [confidence: 60%]

⚔️   Debate round 2/2
────────────────────────────────────────────────────────────
  ⏳ Llama-3 critiquing peers… done  [confidence: 60%]
  ⏳ Mixtral critiquing peers… done  [confidence: 60%]
  ⏳ Gemma-2 critiquing peers… done  [confidence: 60%]

════════════════════════════════════════════════════════════
✅  BEST ANSWER  (from Llama-3)
═══════════════════════════════